# 07.7 — Prompting and Few-Shot Learning

**GPT-3's key finding:** With enough scale, a language model can solve new tasks from a few examples in the prompt — no gradient updates required. This is **in-context learning (ICL)**.

**Why it works (hypothesis):** During pre-training on vast text, the model sees thousands of examples of task demonstrations (Q&A, translations, summaries). It learns to recognize the pattern and continue it — meta-learning from the corpus.

**Prompting paradigms:**
- Zero-shot: task description only
- Few-shot: task description + k examples + new query
- Chain-of-thought (CoT): include reasoning steps in examples
- Self-consistency: generate multiple CoT paths, majority vote

---

In [ ]:
import re
import json
from typing import List, Dict, Optional
from dataclasses import dataclass

# ---- Prompt Templates ----
# Prompts are programs. Think of them as code with a different syntax.

@dataclass
class Example:
    input: str
    output: str
    reasoning: Optional[str] = None  # for chain-of-thought


class FewShotPrompt:
    """
    Build few-shot prompts with optional chain-of-thought.
    
    Format:
    [System instruction]
    
    Input: {example_1_input}
    Output: {example_1_output}
    
    ...more examples...
    
    Input: {query}
    Output:
    """
    def __init__(self, task_description: str, 
                 input_label="Input", output_label="Output"):
        self.task_description = task_description
        self.input_label = input_label
        self.output_label = output_label
        self.examples: List[Example] = []
    
    def add_example(self, input_text: str, output_text: str, reasoning: str = None):
        self.examples.append(Example(input_text, output_text, reasoning))
        return self
    
    def build(self, query: str, cot: bool = False) -> str:
        parts = [self.task_description, ""]
        for ex in self.examples:
            parts.append(f"{self.input_label}: {ex.input}")
            if cot and ex.reasoning:
                parts.append(f"Reasoning: {ex.reasoning}")
            parts.append(f"{self.output_label}: {ex.output}")
            parts.append("")
        parts.append(f"{self.input_label}: {query}")
        parts.append(f"{self.output_label}:")
        return "\n".join(parts)


# Example 1: sentiment classification
sentiment_prompt = FewShotPrompt(
    "Classify the sentiment of the text as Positive, Negative, or Neutral."
)
sentiment_prompt.add_example(
    "The product works great and arrived on time!", "Positive"
).add_example(
    "Terrible quality, broke after two days.", "Negative"
).add_example(
    "The package was delivered.", "Neutral"
)

query = "The food was okay, nothing special but not bad either."
print("=== Few-shot Sentiment Prompt ===")
print(sentiment_prompt.build(query))

In [ ]:
# ---- Chain-of-Thought Prompting ----
# Wei et al. (2022): including step-by-step reasoning in examples
# dramatically improves multi-step reasoning tasks.

math_prompt = FewShotPrompt(
    "Solve the math word problem. Think step by step.",
    input_label="Problem",
    output_label="Answer"
)
math_prompt.add_example(
    input_text="Sarah has 5 apples. She gives 2 to Tom and buys 3 more. How many does she have?",
    output_text="6",
    reasoning="Sarah starts with 5. Gives away 2: 5-2=3. Buys 3 more: 3+3=6."
).add_example(
    input_text="A train travels 60 mph for 2.5 hours. How far does it go?",
    output_text="150 miles",
    reasoning="Distance = speed × time = 60 × 2.5 = 150 miles."
)

query = "A store sells shirts for $25 each. If Jake buys 4 shirts and pays with $120, how much change does he get?"

print("=== Zero-shot (no CoT) ===")
print(math_prompt.build(query, cot=False))
print()
print("=== Chain-of-Thought ===")
print(math_prompt.build(query, cot=True))

print()
print("CoT answer: 4 × $25 = $100. Change = $120 - $100 = $20.")
print("Without CoT: GPT-3-scale models often get this wrong.")
print("With CoT: accuracy on math benchmarks improves ~5-10× on 100B+ models.")
print("On small models (<10B): CoT hurts — they don't have the capacity for it.")

In [ ]:
# ---- Self-consistency decoding ----
# Wang et al. (2022): sample multiple reasoning paths, take majority vote on answer.
# Much more robust than greedy/beam decoding for reasoning tasks.

import random
from collections import Counter

def self_consistency_vote(model_outputs: List[str], 
                           answer_extractor=None) -> tuple:
    """
    Given multiple sampled outputs, extract answers and majority-vote.
    
    model_outputs: list of LLM completions (each has different reasoning chain)
    answer_extractor: callable to pull final answer from text
    Returns: (majority_answer, vote_counts)
    """
    if answer_extractor:
        answers = [answer_extractor(out) for out in model_outputs]
    else:
        # Default: take last word/number as answer
        answers = [out.strip().split()[-1] for out in model_outputs]
    
    counts = Counter(answers)
    majority = counts.most_common(1)[0][0]
    return majority, counts


# Simulate: 5 sampled reasoning paths, mostly correct
simulated_outputs = [
    "Total cost = 4 × 25 = 100. Change = 120 - 100 = $20",
    "4 shirts × $25 = $100. Jake gets back 120 - 100 = $20",
    "Shirts cost 4*25=100 dollars. Change: 120-100 = $20",
    "Each shirt $25. 4 shirts = $90 (error). Change = $30",  # wrong path
    "25 × 4 = 100. 120 - 100 = 20 dollars change",
]

def extract_dollar_amount(text):
    matches = re.findall(r'\$?(\d+)', text)
    return matches[-1] if matches else "unknown"

answer, votes = self_consistency_vote(simulated_outputs, extract_dollar_amount)
print("Self-consistency voting:")
for ans, count in votes.most_common():
    print(f"  ${ans}: {count} votes  {'<-- majority' if ans == answer else ''}")
print(f"Final answer: ${answer}")

In [ ]:
# ---- Structured output prompting ----
# Getting LLMs to return parseable JSON, not free text.

def structured_extraction_prompt(text: str, schema: dict) -> str:
    """
    Prompt for extracting structured information from text.
    
    Works by:
    1. Describing the schema
    2. Few-shot examples with valid JSON
    3. Asking for JSON completion
    """
    schema_str = json.dumps(schema, indent=2)
    prompt = f"""Extract information from the text and return valid JSON matching this schema:
{schema_str}

Rules:
- Only use information from the text
- Use null for missing fields
- Return only the JSON, no explanation

Example:
Text: "John Smith, 34, works as a software engineer at Google in San Francisco."
JSON: {{"name": "John Smith", "age": 34, "occupation": "software engineer", "company": "Google", "city": "San Francisco"}}

Text: "{text}"
JSON:"""
    return prompt


schema = {
    "name": "string",
    "age": "integer or null",
    "occupation": "string or null",
    "company": "string or null",
    "city": "string or null"
}

text = "Dr. Sarah Chen, a 41-year-old cardiologist, recently joined Memorial Hospital in Boston."
print(structured_extraction_prompt(text, schema))

print()
print("Expected output:")
print(json.dumps({"name": "Sarah Chen", "age": 41, 
                  "occupation": "cardiologist", 
                  "company": "Memorial Hospital", "city": "Boston"}, indent=2))

In [ ]:
# ---- Prompt engineering best practices ----

print("=== Prompt Engineering Principles ===")
print()

principles = [
    ("Be specific",
     "'Summarize in 3 bullet points, each under 20 words' beats 'summarize'."),
    
    ("Persona/role",
     "'You are an expert cardiologist' improves medical question answering."),
    
    ("Format examples",
     "Show the exact output format you want. LLMs pattern-match better than they follow descriptions."),
    
    ("Separator tokens",
     "Use ###, ---, or XML tags to delimit sections. Prevents prompt injection."),
    
    ("CoT trigger",
     "'Think step by step' or 'Let's think carefully' → better reasoning."),
    
    ("Negative instructions",
     "'Do NOT include preamble' works less well than 'Start with the answer directly'."),
    
    ("Ordering matters",
     "In few-shot, put the hardest/most representative example last (recency bias)."),
    
    ("Temperature control",
     "Factual extraction: T=0. Creative tasks: T=0.7-1.0. Math: T=0, self-consistency."),
]

for name, tip in principles:
    print(f"  {name:25}: {tip}")

print()
print("=== Prompt injection defense ===")
print()
print("Attack: user input contains 'Ignore previous instructions and...'")
print("Defense options:")
print("  1. Delimiter: wrap user input in XML: <user_input>...</user_input>")
print("  2. Input validation: block known injection patterns")
print("  3. Output filtering: validate output before using it")
print("  4. Separate system/user turns (chat API vs completion API)")
print("  5. Constrained generation: JSON mode, function calling")

print()
print("=== When to fine-tune vs prompt ===")
print()
print("Prompt is enough when:")
print("  - Task is well within LLM capabilities")
print("  - Few-shot examples fit in context window")
print("  - Latency/cost of longer prompts is acceptable")
print()
print("Fine-tune when:")
print("  - Consistent output format required (e.g., specific JSON schema)")
print("  - Few-shot samples don't generalize (edge cases)")
print("  - Need to inject knowledge not in base model")
print("  - Latency: shorter prompts = faster inference")

In [ ]:
# ---- In-context learning: why does it work? ----
# Min et al. (2022): surprising findings

print("=== Why does few-shot ICL work? ===")
print()
print("Min et al. 2022: 'Rethinking the Role of Demonstrations'")
print()
print("Experiment: replace labels in few-shot examples with RANDOM labels.")
print("Result: performance barely drops!")
print()
print("What the demonstrations actually teach the model:")
print("  1. Input distribution (what kinds of inputs to expect)")
print("  2. Output format/space (what the output looks like)")
print("  3. Task structure (the pattern: input -> output)")
print("  NOT: the actual input-output mapping (label content)!")
print()
print("Implications:")
print("  - You need few-shot examples to establish FORMAT, not teach semantics")
print("  - Example quality matters less than example diversity")
print("  - But for complex reasoning, label correctness DOES matter (CoT)")
print()
print("Retrieval-augmented prompting:")
print("  Instead of fixed examples, retrieve the k most similar examples")
print("  to the query from a labeled pool.")
print("  This consistently beats random few-shot selection.")
print()

# Simulate: select examples by similarity to query
def retrieve_similar_examples(query: str, examples: List[Example], k=3):
    """
    Simple word-overlap similarity for illustration.
    In production: use sentence embeddings.
    """
    query_words = set(query.lower().split())
    scored = []
    for ex in examples:
        ex_words = set(ex.input.lower().split())
        overlap = len(query_words & ex_words) / (len(query_words | ex_words) + 1e-9)
        scored.append((ex, overlap))
    scored.sort(key=lambda x: -x[1])
    return [ex for ex, _ in scored[:k]]

pool = [
    Example("The movie was boring and predictable.", "Negative"),
    Example("Great acting, loved every minute.", "Positive"),
    Example("Package arrived yesterday.", "Neutral"),
    Example("The film had poor special effects.", "Negative"),
    Example("Outstanding performance by the cast!", "Positive"),
]

query = "The film had some good moments but dragged in the middle."
retrieved = retrieve_similar_examples(query, pool, k=3)
print(f"Query: '{query}'")
print("Retrieved examples:")
for ex in retrieved:
    print(f"  '{ex.input}' -> {ex.output}")

## Summary — Module 07 Complete

You've covered the full modern NLP stack:
1. BERT architecture & pre-training objectives
2. Fine-tuning for classification, NER
3. Sentence embeddings + FAISS for semantic search
4. DPR: bi-encoder dense retrieval
5. RAG: end-to-end retrieval-augmented generation
6. LoRA: parameter-efficient fine-tuning
7. Prompting: few-shot, CoT, self-consistency, structured output

**Key insight:** Modern NLP is a stack of composable pieces. RAG = retrieval (DPR/BM25) + generation (LLM). Fine-tuning = BERT + task head or LLM + LoRA. Evaluation is the glue that tells you which combination works.

---
**Module 08: Evaluation & Production**